In [35]:
import pandas as pd

matches = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

def process_matches(df):
    #explore the data 
    print(df.shape)
    df.head(5)
    print(df.isnull().sum())
    print(df.dtypes)
    print(df.columns)
    #clean
    df = df.fillna({'city':'unknown','player_of_match':'unknown','winner':'no_result'})
    print(df[['city','player_of_match','result_margin','winner','method']].isnull().sum())
    #transform
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month

    return df

def process_deliveries(df):
    #explore the data
    print()
    print("the deliveries data")
    print(df.shape)
    print(df.isnull().sum())
    print(df.dtypes)
    print(df[['match_id','inning','batting_team','bowling_team','over','batter','bowler','extras_type','player_dismissed','dismissal_kind','fielder']])
    #clean
    #transform
    df['is_extra'] = df['extras_type'].notna().astype(int)
    
    return df
    

clean_matches =   process_matches(matches)
clean_deliveries =  process_deliveries(deliveries)   


(1095, 20)
id                    0
season                0
city                 51
date                  0
match_type            0
player_of_match       5
venue                 0
team1                 0
team2                 0
toss_winner           0
toss_decision         0
winner                5
result                0
result_margin        19
target_runs           3
target_overs          3
super_over            0
method             1074
umpire1               0
umpire2               0
dtype: int64
id                   int64
season              object
city                object
date                object
match_type          object
player_of_match     object
venue               object
team1               object
team2               object
toss_winner         object
toss_decision       object
winner              object
result              object
result_margin      float64
target_runs        float64
target_overs       float64
super_over          object
method              object
umpire1   

In [40]:
import sqlite3

conn = sqlite3.connect("ipl_data.db")

clean_matches.to_sql("matches",conn,if_exists="replace", index=False)
clean_deliveries.to_sql("deliveries",conn,if_exists="replace", index=False)
print()
print("Data loaded successfully!")
print("matches : " , pd.read_sql("select count(*) as total from matches ",conn).iloc[0,0])
print("deliveries : " , pd.read_sql("select count(*) as total from deliveries",conn).iloc[0,0])



Data loaded successfully!
matches :  1095
deliveries :  260920


In [45]:
# Top 10 wicket takers
sql_1 = """ 
select * from (select bowler ,count(is_wicket) as wickets from deliveries
where is_wicket = 1
group by bowler) order by wickets desc limit 10 
"""
print("Top 10 wicket takers:")
print(pd.read_sql(sql_1,conn))

Top 10 wicket takers:
       bowler  wickets
0   YS Chahal      213
1    DJ Bravo      207
2   PP Chawla      201
3   SP Narine      200
4    R Ashwin      198
5     B Kumar      195
6  SL Malinga      188
7    A Mishra      183
8   JJ Bumrah      182
9   RA Jadeja      169


In [57]:
#Matches per season
sql_2 = """ 
select year , count(*) as matches from matches group by year ;
"""
print(pd.read_sql(sql_2,conn))

    year  matches
0   2008       58
1   2009       57
2   2010       60
3   2011       73
4   2012       74
5   2013       76
6   2014       60
7   2015       59
8   2016       60
9   2017       59
10  2018       60
11  2019       60
12  2020       60
13  2021       60
14  2022       74
15  2023       74
16  2024       71


In [54]:
# Best venue 
sql_3 = """ 
select venue ,count(*) as matches_played from 
matches 
group by venue 
order by count(*) desc
limit 10
"""
print(pd.read_sql(sql_3,conn))

          venue  matches_played
0  Eden Gardens              77


In [66]:
#Batting first vs second wins
sql_4 = """select toss_decision ,count(*) as total ,
sum(case when toss_winner = winner then 1 else 0 end) as win,
round(sum(case when toss_winner = winner then 1 else 0 end)*100/count(*),2) as winpercent
from matches 
where winner <> 'no_result' 
group by toss_decision  
"""
print(pd.read_sql(sql_4,conn))

  toss_decision  total  win  winpercent
0           bat    390  177        45.0
1         field    700  377        53.0


In [70]:
# most player of the match awards
sql_5 = """  
select player_of_match,count(*) total 
from matches group by player_of_match order by total desc
limit 10
"""
print(pd.read_sql(sql_5,conn))


  player_of_match  total
0  AB de Villiers     25
1        CH Gayle     22
2       RG Sharma     19
3         V Kohli     18
4       DA Warner     18
5        MS Dhoni     17
6       YK Pathan     16
7       SR Watson     16
8       RA Jadeja     16
9       SP Narine     15


In [74]:
#Highest scoring overs (which over has most runs)
sql_6 = """  
select over,sum(total_runs) as runs from deliveries group by over order by runs desc limit 6
"""
print(pd.read_sql(sql_6,conn))


   over      runs
0    19  1.776855
1    18  1.646896
2    17  1.587839
3    16  1.498778
4    15  1.434273
5    14  1.393504


In [77]:
#Most extras given by a team (most expensive team)
sql_7 = """  
select bowling_team as team,sum(extra_runs) as extra_runs from deliveries group by bowling_team 
order by extra_runs desc limit 5
"""
print(pd.read_sql(sql_7,conn))

                          team  extra_runs
0               Mumbai Indians        2295
1  Royal Challengers Bangalore        2040
2        Kolkata Knight Riders        1957
3             Rajasthan Royals        1917
4          Chennai Super Kings        1842
